In [1]:
%load_ext cudf.pandas

In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
import warnings
import optuna
import gc
from sklearn.model_selection import StratifiedKFold
from pandas.errors import PerformanceWarning
from sklearn.metrics import roc_auc_score
from optuna.samplers import TPESampler
from itertools import combinations
from xgboost import XGBClassifier
from tqdm import tqdm

warnings.simplefilter(action="ignore", category=PerformanceWarning)
TARGET = 'y'
NUMS = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
CATS = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s5e8/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s5e8/test.csv', index_col='id')
orig = pd.read_csv('/kaggle/input/bank-marketing-dataset-full/bank-full.csv', delimiter=';')
orig['y'] = orig['y'].replace({'yes': 1, 'no': 0})

train[CATS] = train[CATS].astype('category')
test[CATS] = test[CATS].astype('category')
orig[CATS] = orig[CATS].astype('category')

TE_columns = []

columns = NUMS + CATS

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)
        
        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]

        TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

/usr/local/lib/python3.11/dist-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return fn(*args, **kwargs)
100%|██████████| 120/120 [00:05<00:00, 20.89it/s]


In [4]:
def target_encode(train, valid, test, col, target=TARGET, kfold=5, smooth=20, agg='mean'):
    train['kfold'] = ((train.index) % kfold)
    col_name = '_'.join(col)
    train[f'TE_{agg.upper()}_' + col_name] = 0.
    for i in range(kfold):
        df_tmp = train[train['kfold'] != i]
        if agg == 'mean': mn = train[target].mean()
        elif agg == 'median': mn = train[target].median()
        elif agg == 'min': mn = train[target].min()
        elif agg == 'max': mn = train[target].max()
        elif agg == 'nunique': mn = 0
        df_tmp = df_tmp[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
        df_tmp.columns = col + [agg, 'count']
        if agg == 'nunique':
            df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
        else:
            df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
        df_tmp_m = train[col + ['kfold', f'TE_{agg.upper()}_' + col_name]].merge(df_tmp, how='left', left_on=col, right_on=col)
        df_tmp_m.loc[df_tmp_m['kfold'] == i, f'TE_{agg.upper()}_' + col_name] = df_tmp_m.loc[df_tmp_m['kfold'] == i, 'TE_tmp']
        train[f'TE_{agg.upper()}_' + col_name] = df_tmp_m[f'TE_{agg.upper()}_' + col_name].fillna(mn).values

    df_tmp = train[col + [target]].groupby(col).agg([agg, 'count']).reset_index()
    if agg == 'mean': mn = train[target].mean()
    elif agg == 'median': mn = train[target].median()
    elif agg == 'min': mn = train[target].min()
    elif agg == 'max': mn = train[target].max()
    elif agg == 'nunique': mn = 0
    df_tmp.columns = col + [agg, 'count']
    if agg == 'nunique':
        df_tmp['TE_tmp'] = df_tmp[agg] / df_tmp['count']
    else:
        df_tmp['TE_tmp'] = ((df_tmp[agg] * df_tmp['count']) + (mn * smooth)) / (df_tmp['count'] + smooth)
    df_tmp_m = valid[col].merge(df_tmp, how='left', left_on=col, right_on=col)
    valid[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
    valid[f'TE_{agg.upper()}_' + col_name] = valid[f'TE_{agg.upper()}_' + col_name].astype('float32')

    df_tmp_m = test[col].merge(df_tmp, how='left', left_on=col, right_on=col)
    test[f'TE_{agg.upper()}_' + col_name] = df_tmp_m['TE_tmp'].fillna(mn).values
    test[f'TE_{agg.upper()}_' + col_name] = test[f'TE_{agg.upper()}_' + col_name].astype('float32')

    train = train.drop('kfold', axis=1)
    train[f'TE_{agg.upper()}_' + col_name] = train[f'TE_{agg.upper()}_' + col_name].astype('float32')

    return (train, valid, test)

def count_encode(train, valid, test, col):
    counts = train[col].value_counts()

    train[f'CE_{col}'] = train[col].map(counts)
    valid[f'CE_{col}'] = valid[col].map(counts).fillna(0)
    test[f'CE_{col}'] = test[col].map(counts).fillna(0)
    return (train, valid, test)

In [5]:
oof = np.zeros(len(train))
pred = np.zeros(len(test))

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

for idx, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET])):
    X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
    y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
    X_test = test.copy()

    X_train = pd.concat([X_train, orig[FEATURES]])
    y_train = pd.concat([y_train, orig[TARGET]])

    for col in tqdm(TE_columns):
        X_train, X_val, X_test = target_encode(pd.concat([X_train, y_train], axis=1), X_val, X_test, [col], smooth=10, agg='mean')
        #X_train, X_val, X_test = target_encode(X_train, X_val, X_test, [col], smooth=10, agg='median')
        X_train = X_train.drop(TARGET, axis=1)
        X_train, X_val, X_test = count_encode(X_train, X_val, X_test, col)
    
        X_train = X_train.drop(col, axis=1)
        X_val = X_val.drop(col, axis=1)
        X_test = X_test.drop(col, axis=1)

    param_grid = {'colsample_bytree': 0.3413131357034518, 
                  'subsample': 0.8991973526634904, 
                  'reg_lambda': 4.063399595895267, 
                  'reg_alpha': 2.917406632455954, 
                  'max_depth': 8}

    model = XGBClassifier(**param_grid, 
                          n_estimators=10000,
                          objective='binary:logistic',
                          eval_metric='auc',
                          learning_rate=0.01,
                          early_stopping_rounds=200,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1]

    print(f'Fold {idx + 1}: {roc_auc_score(y_val, oof[val_idx])}')

    del model, X_train, X_val, y_train, y_val, X_test
    gc.collect()

pred /= 5
print(f'CV AUC: {roc_auc_score(train[TARGET], oof)}')

100%|██████████| 120/120 [01:25<00:00,  1.40it/s]


[0]	validation_0-auc:0.95952
[100]	validation_0-auc:0.97158
[200]	validation_0-auc:0.97265
[300]	validation_0-auc:0.97343
[400]	validation_0-auc:0.97402
[500]	validation_0-auc:0.97445
[600]	validation_0-auc:0.97478
[700]	validation_0-auc:0.97504
[800]	validation_0-auc:0.97527
[900]	validation_0-auc:0.97543
[1000]	validation_0-auc:0.97556
[1100]	validation_0-auc:0.97568
[1200]	validation_0-auc:0.97578
[1300]	validation_0-auc:0.97588
[1400]	validation_0-auc:0.97594
[1500]	validation_0-auc:0.97600
[1600]	validation_0-auc:0.97606
[1700]	validation_0-auc:0.97610
[1800]	validation_0-auc:0.97614
[1900]	validation_0-auc:0.97617
[2000]	validation_0-auc:0.97621
[2100]	validation_0-auc:0.97624
[2200]	validation_0-auc:0.97627
[2300]	validation_0-auc:0.97629
[2400]	validation_0-auc:0.97631
[2500]	validation_0-auc:0.97633
[2600]	validation_0-auc:0.97635
[2700]	validation_0-auc:0.97636
[2800]	validation_0-auc:0.97638
[2900]	validation_0-auc:0.97639
[3000]	validation_0-auc:0.97641
[3100]	validation_0-

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:48:15] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Fold 1: 0.9764632112174711


100%|██████████| 120/120 [01:26<00:00,  1.39it/s]


...

[0]	validation_0-auc:0.95855
[100]	validation_0-auc:0.96983
[200]	validation_0-auc:0.97090
[300]	validation_0-auc:0.97170
[400]	validation_0-auc:0.97232
[500]	validation_0-auc:0.97280
[600]	validation_0-auc:0.97315
[700]	validation_0-auc:0.97342
[800]	validation_0-auc:0.97368
[900]	validation_0-auc:0.97385
[1000]	validation_0-auc:0.97401
[1100]	validation_0-auc:0.97414
[1200]	validation_0-auc:0.97425
[1300]	validation_0-auc:0.97435
[1400]	validation_0-auc:0.97444
[1500]	validation_0-auc:0.97449
[1600]	validation_0-auc:0.97455
[1700]	validation_0-auc:0.97459
[1800]	validation_0-auc:0.97463
[1900]	validation_0-auc:0.97466
[2000]	validation_0-auc:0.97469
[2100]	validation_0-auc:0.97472
[2200]	validation_0-auc:0.97475
[2300]	validation_0-auc:0.97477
[2400]	validation_0-auc:0.97479
[2500]	validation_0-auc:0.97481
[2600]	validation_0-auc:0.97482
[2700]	validation_0-auc:0.97484
[2800]	validation_0-auc:0.97486
[2900]	validation_0-auc:0.97487
[3000]	validation_0-auc:0.97488
[3100]	validation_0-

100%|██████████| 120/120 [01:26<00:00,  1.39it/s]


[0]	validation_0-auc:0.96088
[100]	validation_0-auc:0.97134
[200]	validation_0-auc:0.97237
[300]	validation_0-auc:0.97309
[400]	validation_0-auc:0.97368
[500]	validation_0-auc:0.97414
[600]	validation_0-auc:0.97444
[700]	validation_0-auc:0.97471
[800]	validation_0-auc:0.97493
[900]	validation_0-auc:0.97510
[1000]	validation_0-auc:0.97525
[1100]	validation_0-auc:0.97539
[1200]	validation_0-auc:0.97549
[1300]	validation_0-auc:0.97559
[1400]	validation_0-auc:0.97566
[1500]	validation_0-auc:0.97572
[1600]	validation_0-auc:0.97579
[1700]	validation_0-auc:0.97583
[1800]	validation_0-auc:0.97587
[1900]	validation_0-auc:0.97591
[2000]	validation_0-auc:0.97595
[2100]	validation_0-auc:0.97598
[2200]	validation_0-auc:0.97602
[2300]	validation_0-auc:0.97604
[2400]	validation_0-auc:0.97605
[2500]	validation_0-auc:0.97607
[2600]	validation_0-auc:0.97609
[2700]	validation_0-auc:0.97611
[2800]	validation_0-auc:0.97612
[2900]	validation_0-auc:0.97613
[3000]	validation_0-auc:0.97614
[3100]	validation_0-

100%|██████████| 120/120 [01:26<00:00,  1.39it/s]


[0]	validation_0-auc:0.95940
[100]	validation_0-auc:0.97063
[200]	validation_0-auc:0.97171
[300]	validation_0-auc:0.97247
[400]	validation_0-auc:0.97308
[500]	validation_0-auc:0.97354
[600]	validation_0-auc:0.97390
[700]	validation_0-auc:0.97415
[800]	validation_0-auc:0.97438
[900]	validation_0-auc:0.97455
[1000]	validation_0-auc:0.97470
[1100]	validation_0-auc:0.97483
[1200]	validation_0-auc:0.97494
[1300]	validation_0-auc:0.97504
[1400]	validation_0-auc:0.97512
[1500]	validation_0-auc:0.97519
[1600]	validation_0-auc:0.97524
[1700]	validation_0-auc:0.97529
[1800]	validation_0-auc:0.97533
[1900]	validation_0-auc:0.97537
[2000]	validation_0-auc:0.97541
[2100]	validation_0-auc:0.97543
[2200]	validation_0-auc:0.97545
[2300]	validation_0-auc:0.97548
[2400]	validation_0-auc:0.97550
[2500]	validation_0-auc:0.97552
[2600]	validation_0-auc:0.97554
[2700]	validation_0-auc:0.97556
[2800]	validation_0-auc:0.97557
[2900]	validation_0-auc:0.97559
[3000]	validation_0-auc:0.97559
[3100]	validation_0-

In [6]:
submission = pd.read_csv('/kaggle/input/playground-series-s5e8/sample_submission.csv')
submission['y'] = pred
submission.to_csv('xgb.csv', index=False)
pd.DataFrame({'xgb_oof': oof}).to_csv('xgb_oof.csv', index=False)
pd.DataFrame({'xgb_pred': pred}).to_csv('xgb_pred.csv', index=False)